# Semantic Mask CutMix

Ex01에서 사용한 DeepLabV3 semantic segmentation을 Stanford Dogs 분류 실험에 연결합니다.
DeepLab이 예측한 `dog` 윤곽만 다른 이미지의 임의 위치에 붙이고, 실제 붙인 mask 면적으로
두 레이블의 Cross Entropy 비율을 계산합니다.

- 분류 모델: ImageNet pretrained ResNet-50
- image augmentation: Basic (horizontal flip 50% + brightness jitter 0.1)
- batch mixing: Semantic Mask CutMix, batch 적용 요청 확률 `p=0.5`
- alpha: 사용하지 않음
- epoch 20, seed 42, SGD, learning rate 0.001

기존 `Basic + CutMix alpha=0.2, p=0.5` 결과는 재학습하지 않고 참고값으로만 읽습니다.

> 최초 실행은 전체 20,580장의 segmentation mask를 생성하므로 시간이 오래 걸립니다.
> 커널을 재시작한 뒤 위에서부터 순서대로 실행하세요.


## 1. 환경 및 경로 설정

mask cache와 Semantic CutMix 결과는 기존 실험과 분리된 폴더에 저장합니다.
DeepLab과 ResNet-50 pretrained weight는 로컬 PyTorch cache를 사용합니다.


In [ ]:
from pathlib import Path
import copy
import gc
import hashlib
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision.datasets import ImageFolder
from torchvision.models.segmentation import (
    DeepLabV3_ResNet101_Weights,
    deeplabv3_resnet101,
)


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
MASK_BATCH_SIZE = 8
LEARNING_RATE = 0.001
MIX_PROBABILITY = 0.5
NUM_WORKERS = 2
COOLDOWN_EVERY_EPOCHS = 5
COOLDOWN_SECONDS = 60
IMAGE_SIZE = (224, 224)
SEGMENTATION_SIZE = (520, 520)
DOG_CLASS_ID = 12

DATASET_DIR = Path("~/datasets/stanfordimagenetdogs/Images").expanduser()
RESULT_DIR = Path("results_resnet50_segcutmix")
MASK_DIR = RESULT_DIR / "masks"
MASK_STATS_PATH = RESULT_DIR / "mask_statistics.csv"
MASK_CONFIG_PATH = RESULT_DIR / "mask_cache_config.json"
HISTORY_PATH = RESULT_DIR / "training_history.csv"
CHECKPOINT_PATH = RESULT_DIR / "semantic_cutmix_best.pt"
EXPERIMENT_CONFIG_PATH = RESULT_DIR / "experiment_config.json"
COMPARISON_PATH = RESULT_DIR / "comparison.png"

REFERENCE_SUMMARY_PATH = Path(
    "results_resnet50_aug_alpha02_p05/experiment_summary.csv"
)
REFERENCE_EXPERIMENT = "basic_cutmix_alpha0.2_p0.5"

# True로 바꾸면 checkpoint가 있어도 Semantic CutMix를 처음부터 다시 학습합니다.
RETRAIN = False

RESULT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f"Device: {DEVICE}")
print(f"Dataset: {DATASET_DIR}")
print(f"Mask cache: {MASK_DIR.resolve()}")
print(f"Results: {RESULT_DIR.resolve()}")

## 2. 데이터 분할과 기본 transform

기존 augmentation 실험과 같은 `58.3% / 41.7%` split과 seed를 사용합니다.
분류 입력은 항상 `224×224`이며 normalization도 기존 실험과 동일합니다.


In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"데이터셋을 찾을 수 없습니다: {DATASET_DIR}")


def classification_transform():
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        ),
    ])


full_dataset = ImageFolder(
    root=DATASET_DIR,
    transform=classification_transform(),
)
total_size = len(full_dataset)
train_size = int(0.583 * total_size)
val_size = total_size - train_size

split_generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=split_generator,
)

class_names = full_dataset.classes
num_classes = len(class_names)


def indices_fingerprint(indices):
    index_array = np.asarray(list(indices), dtype=np.int64)
    return hashlib.sha256(index_array.tobytes()).hexdigest()


train_indices_sha256 = indices_fingerprint(train_subset.indices)
val_indices_sha256 = indices_fingerprint(val_subset.indices)

print(f"Classes: {num_classes}")
print(f"Total: {total_size:,}")
print(f"Train: {len(train_subset):,} | Validation: {len(val_subset):,}")
print(f"Train indices SHA256: {train_indices_sha256}")
print(f"Validation indices SHA256: {val_indices_sha256}")

## 3. DeepLab dog mask cache 생성

Ex01과 같은 DeepLabV3-ResNet101을 사용합니다.

1. 이미지를 `520×520`으로 변환하고 ImageNet normalization을 적용
2. 각 픽셀에서 가장 높은 semantic class를 `argmax`
3. VOC `dog` 클래스 ID 12만 binary mask로 선택
4. nearest-neighbor로 `224×224`에 맞춘 뒤 PNG로 저장

cache 설정이 현재 코드와 다르면 기존 mask를 자동 재사용하지 않고 오류를 발생시킵니다.


In [ ]:
segmentation_weights = DeepLabV3_ResNet101_Weights.DEFAULT
segmentation_categories = segmentation_weights.meta["categories"]
assert segmentation_categories[DOG_CLASS_ID] == "dog"

mask_cache_config = {
    "model": "deeplabv3_resnet101",
    "weights": segmentation_weights.name,
    "dog_class_id": DOG_CLASS_ID,
    "dog_class_name": segmentation_categories[DOG_CLASS_ID],
    "segmentation_size": list(SEGMENTATION_SIZE),
    "mask_size": list(IMAGE_SIZE),
    "dataset_dir": str(DATASET_DIR.resolve()),
    "dataset_samples": total_size,
}

if MASK_CONFIG_PATH.exists():
    saved_mask_config = json.loads(
        MASK_CONFIG_PATH.read_text(encoding="utf-8")
    )
    if saved_mask_config != mask_cache_config:
        raise RuntimeError(
            "기존 mask cache 설정이 현재 설정과 다릅니다. "
            f"{MASK_DIR}와 {MASK_CONFIG_PATH}를 삭제한 뒤 다시 실행하세요."
        )
else:
    MASK_CONFIG_PATH.write_text(
        json.dumps(mask_cache_config, indent=2),
        encoding="utf-8",
    )


def mask_path_for_index(dataset_index):
    return MASK_DIR / f"{dataset_index:05d}.png"


segmentation_transform = transforms.Compose([
    transforms.Resize(SEGMENTATION_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class MaskGenerationDataset(Dataset):
    def __init__(self, samples, dataset_indices):
        self.samples = samples
        self.dataset_indices = list(dataset_indices)

    def __len__(self):
        return len(self.dataset_indices)

    def __getitem__(self, item):
        dataset_index = self.dataset_indices[item]
        image_path, _ = self.samples[dataset_index]
        image = Image.open(image_path).convert("RGB")
        return segmentation_transform(image), dataset_index


missing_mask_indices = [
    index
    for index in range(total_size)
    if not mask_path_for_index(index).exists()
]

print(f"Cached masks: {total_size - len(missing_mask_indices):,}")
print(f"Masks to generate: {len(missing_mask_indices):,}")

In [ ]:
if missing_mask_indices:
    segmentation_model = deeplabv3_resnet101(
        weights=segmentation_weights
    ).to(DEVICE)
    segmentation_model.eval()

    mask_loader = DataLoader(
        MaskGenerationDataset(
            full_dataset.samples,
            missing_mask_indices,
        ),
        batch_size=MASK_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    with torch.no_grad():
        for batch_index, (images, dataset_indices) in enumerate(mask_loader, 1):
            images = images.to(DEVICE, non_blocking=True)
            logits = segmentation_model(images)["out"]
            predictions = logits.argmax(dim=1)
            dog_masks = predictions.eq(DOG_CLASS_ID).float().unsqueeze(1)
            dog_masks = F.interpolate(
                dog_masks,
                size=IMAGE_SIZE,
                mode="nearest",
            ).squeeze(1).bool().cpu().numpy()

            for dog_mask, dataset_index in zip(
                dog_masks,
                dataset_indices.tolist(),
            ):
                mask_image = Image.fromarray(
                    dog_mask.astype(np.uint8) * 255,
                    mode="L",
                )
                mask_image.save(mask_path_for_index(dataset_index))

            if batch_index % 100 == 0 or batch_index == len(mask_loader):
                completed = min(
                    batch_index * MASK_BATCH_SIZE,
                    len(missing_mask_indices),
                )
                print(
                    f"Mask generation: {completed:,}/"
                    f"{len(missing_mask_indices):,}"
                )

    segmentation_model.to("cpu")
    del segmentation_model, mask_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("모든 mask가 cache되어 있어 segmentation 추론을 건너뜁니다.")

missing_after_generation = [
    index
    for index in range(total_size)
    if not mask_path_for_index(index).exists()
]
assert not missing_after_generation

mask_rows = []
total_pixels = IMAGE_SIZE[0] * IMAGE_SIZE[1]
for dataset_index, (image_path, label) in enumerate(full_dataset.samples):
    mask = np.asarray(
        Image.open(mask_path_for_index(dataset_index)).convert("L")
    ) > 0
    mask_pixels = int(mask.sum())
    mask_rows.append({
        "dataset_index": dataset_index,
        "image_path": image_path,
        "label": label,
        "class_name": class_names[label],
        "mask_pixels": mask_pixels,
        "mask_ratio": mask_pixels / total_pixels,
        "detected": mask_pixels > 0,
    })

mask_stats_df = pd.DataFrame(mask_rows)
mask_stats_df.to_csv(MASK_STATS_PATH, index=False)

detected_count = int(mask_stats_df["detected"].sum())
detected_ratio = mask_stats_df["detected"].mean()
detected_masks = mask_stats_df.loc[
    mask_stats_df["detected"],
    "mask_ratio",
]

print(
    f"Dog mask detected: {detected_count:,}/{total_size:,} "
    f"({100.0 * detected_ratio:.2f}%)"
)
print(
    f"Detected mask ratio | mean={detected_masks.mean():.4f}, "
    f"std={detected_masks.std():.4f}, "
    f"min={detected_masks.min():.4f}, "
    f"max={detected_masks.max():.4f}"
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(detected_masks, bins=40, color="tab:blue", alpha=0.8)
ax.set_title("DeepLab Dog Mask Area Distribution")
ax.set_xlabel("Dog mask pixels / image pixels")
ax.set_ylabel("Images")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 4. 이미지와 mask가 함께 움직이는 데이터셋

horizontal flip은 이미지와 mask에 동시에 적용합니다. Brightness jitter는 픽셀 색만
바꾸므로 이미지에만 적용합니다. validation에는 augmentation을 적용하지 않습니다.


In [ ]:
class CachedMaskDogDataset(Dataset):
    def __init__(self, samples, indices, train=False):
        self.samples = samples
        self.indices = list(indices)
        self.train = train
        self.brightness_jitter = transforms.ColorJitter(brightness=0.1)
        self.to_classifier_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5],
            ),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        dataset_index = self.indices[item]
        image_path, label = self.samples[dataset_index]

        image = Image.open(image_path).convert("RGB")
        image = TF.resize(image, IMAGE_SIZE)
        mask = Image.open(mask_path_for_index(dataset_index)).convert("L")
        if mask.size != (IMAGE_SIZE[1], IMAGE_SIZE[0]):
            mask = TF.resize(
                mask,
                IMAGE_SIZE,
                interpolation=transforms.InterpolationMode.NEAREST,
            )

        if self.train and torch.rand(1).item() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        if self.train:
            image = self.brightness_jitter(image)

        image_tensor = self.to_classifier_tensor(image)
        mask_tensor = torch.from_numpy(
            (np.asarray(mask) > 0).copy()
        ).bool()

        assert image_tensor.shape == (3, *IMAGE_SIZE)
        assert mask_tensor.shape == IMAGE_SIZE
        return image_tensor, label, mask_tensor


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_loader(dataset, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        worker_init_fn=seed_worker,
    )


train_dataset = CachedMaskDogDataset(
    full_dataset.samples,
    train_subset.indices,
    train=True,
)
val_dataset = CachedMaskDogDataset(
    full_dataset.samples,
    val_subset.indices,
    train=False,
)

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)

sample_images, sample_labels, sample_masks = next(iter(train_loader))
assert sample_images.shape[1:] == (3, *IMAGE_SIZE)
assert sample_masks.shape[1:] == IMAGE_SIZE
print(f"Image batch: {sample_images.shape}")
print(f"Mask batch: {sample_masks.shape}")

## 5. Semantic Mask CutMix

선택된 batch의 각 receiver는 다른 sample의 dog mask crop을 받습니다.
bounding box는 객체 crop을 추출하는 데만 사용하고 실제 교체는 mask가 `True`인 강아지
윤곽 픽셀에만 수행합니다.

sample마다 mask 크기가 다르므로 `lam`도 sample별로 계산합니다. batch 안에 mask가 없는
donor가 하나라도 있으면 그 batch 전체의 mixing을 건너뜁니다.


In [ ]:
def semantic_cutmix_batch(images, labels, masks):
    batch_size, _, height, width = images.shape
    if batch_size < 2:
        ones = torch.ones(batch_size, device=images.device)
        return images, labels, labels, ones, False

    # roll offset이 1 이상이므로 자기 자신을 donor로 선택하지 않습니다.
    offset = int(torch.randint(1, batch_size, (1,), device=images.device))
    donor_indices = torch.roll(
        torch.arange(batch_size, device=images.device),
        shifts=offset,
    )
    donor_masks = masks[donor_indices]

    # 계획대로 donor mask가 하나라도 없으면 batch 전체를 일반 CE로 처리합니다.
    if not donor_masks.flatten(1).any(dim=1).all():
        ones = torch.ones(batch_size, device=images.device)
        return images, labels, labels, ones, False

    mixed_images = images.clone()
    receiver_lams = torch.ones(batch_size, device=images.device)

    for receiver_index in range(batch_size):
        donor_index = donor_indices[receiver_index]
        donor_mask = masks[donor_index]
        coordinates = donor_mask.nonzero(as_tuple=False)

        y1, x1 = coordinates.min(dim=0).values
        y2, x2 = coordinates.max(dim=0).values + 1

        donor_crop = images[donor_index, :, y1:y2, x1:x2]
        mask_crop = donor_mask[y1:y2, x1:x2]
        crop_height, crop_width = mask_crop.shape

        max_top = height - crop_height
        max_left = width - crop_width
        paste_top = int(
            torch.randint(max_top + 1, (1,), device=images.device)
        )
        paste_left = int(
            torch.randint(max_left + 1, (1,), device=images.device)
        )

        target_region = mixed_images[
            receiver_index,
            :,
            paste_top:paste_top + crop_height,
            paste_left:paste_left + crop_width,
        ]
        target_region[:, mask_crop] = donor_crop[:, mask_crop]

        donor_ratio = mask_crop.sum().float() / (height * width)
        receiver_lams[receiver_index] = 1.0 - donor_ratio

    assert mixed_images.shape == images.shape
    assert torch.all((0.0 <= receiver_lams) & (receiver_lams <= 1.0))
    assert torch.allclose(
        receiver_lams + (1.0 - receiver_lams),
        torch.ones_like(receiver_lams),
    )

    return (
        mixed_images,
        labels,
        labels[donor_indices],
        receiver_lams,
        True,
    )

## 6. Mask와 Semantic Mask CutMix 시각화

`node-2-final.ipynb`의 MixUp/CutMix 비교처럼 서로 다른 견종 네 장을 선택해 두 쌍을
비교합니다. 빨간 영역은 DeepLab이 dog로 예측한 mask이며, `Extracted donor`는 실제로
receiver에 붙는 donor의 강아지 픽셀만 표시합니다.

시각화용 난수는 `torch.random.fork_rng()` 안에서만 사용하므로 이후 학습의 random
state에는 영향을 주지 않습니다.


In [ ]:
successful_train_rows = mask_stats_df.loc[
    mask_stats_df["dataset_index"].isin(train_subset.indices)
    & mask_stats_df["detected"]
].copy()

# node-2-final과 같이 서로 다른 label의 source 네 장을 선택합니다.
visualization_indices = (
    successful_train_rows
    .drop_duplicates(subset="label")
    .head(4)["dataset_index"]
    .astype(int)
    .tolist()
)
if len(visualization_indices) < 4:
    raise RuntimeError(
        "서로 다른 label을 가진 mask 검출 성공 이미지 네 장이 필요합니다."
    )


def load_visualization_sample(dataset_index):
    image_path, label = full_dataset.samples[dataset_index]
    image = TF.resize(
        Image.open(image_path).convert("RGB"),
        IMAGE_SIZE,
    )
    mask = TF.resize(
        Image.open(mask_path_for_index(dataset_index)).convert("L"),
        IMAGE_SIZE,
        interpolation=transforms.InterpolationMode.NEAREST,
    )
    mask_array = np.asarray(mask) > 0
    return image, label, mask_array


def add_mask_overlay(image, mask, color=(255, 0, 0), alpha=0.45):
    image_array = np.asarray(image).copy()
    color_array = np.asarray(color, dtype=np.float32)
    image_array[mask] = (
        (1.0 - alpha) * image_array[mask] + alpha * color_array
    ).astype(np.uint8)
    return image_array


def extract_masked_object(image, mask):
    extracted = np.zeros((*IMAGE_SIZE, 3), dtype=np.uint8)
    image_array = np.asarray(image)
    extracted[mask] = image_array[mask]
    return extracted


def denormalize(image_tensor):
    return torch.clamp(image_tensor * 0.5 + 0.5, 0, 1)


visual_samples = [
    load_visualization_sample(index)
    for index in visualization_indices
]
pair_indices = [(0, 1), (2, 3)]

fig, axes = plt.subplots(
    len(pair_indices),
    6,
    figsize=(24, 8),
    squeeze=False,
)

for row, (receiver_index, donor_index) in enumerate(pair_indices):
    receiver_image, receiver_label, receiver_mask = visual_samples[
        receiver_index
    ]
    donor_image, donor_label, donor_mask = visual_samples[donor_index]

    image_batch = torch.stack([
        classification_transform()(receiver_image),
        classification_transform()(donor_image),
    ])
    mask_batch = torch.from_numpy(
        np.stack([receiver_mask, donor_mask])
    ).bool()
    label_batch = torch.tensor(
        [receiver_label, donor_label],
        dtype=torch.long,
    )

    # batch size가 2이므로 receiver 0의 donor는 항상 sample 1입니다.
    with torch.random.fork_rng():
        torch.manual_seed(SEED + 1000 + row)
        mixed_batch, _, donor_labels, visual_lams, applied = (
            semantic_cutmix_batch(
                image_batch,
                label_batch,
                mask_batch,
            )
        )
    assert applied
    assert donor_labels[0].item() == donor_label

    donor_ratio = 1.0 - visual_lams[0].item()
    display_items = [
        (
            np.asarray(receiver_image),
            f"Source {chr(65 + receiver_index)}\n"
            f"{class_names[receiver_label]}",
        ),
        (
            add_mask_overlay(receiver_image, receiver_mask),
            "Receiver + dog mask",
        ),
        (
            np.asarray(donor_image),
            f"Source {chr(65 + donor_index)}\n"
            f"{class_names[donor_label]}",
        ),
        (
            add_mask_overlay(donor_image, donor_mask),
            "Donor + dog mask",
        ),
        (
            extract_masked_object(donor_image, donor_mask),
            f"Extracted donor\nratio={donor_ratio:.3f}",
        ),
        (
            denormalize(mixed_batch[0]).permute(1, 2, 0).numpy(),
            (
                f"Mask CutMix: {class_names[receiver_label]}\n"
                f"+ {class_names[donor_label]} {donor_ratio:.3f}"
            ),
        ),
    ]

    for col, (item, title) in enumerate(display_items):
        axes[row, col].imshow(item)
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(
        f"Comparison {row + 1}",
        fontsize=12,
    )

fig.suptitle(
    "DeepLab dog masks and Semantic Mask CutMix examples",
    fontsize=15,
)
fig.tight_layout()
plt.show()


## 7. ResNet-50 초기값과 기존 실험 hash 검증

ImageNet pretrained ResNet-50과 새 FC layer를 한 번 초기화합니다. 기존 p=0.5 history가
있으면 초기 모델과 데이터 분할 hash가 동일한지 확인합니다.


In [ ]:
set_seed()
base_model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V1
)
base_model.fc = nn.Linear(base_model.fc.in_features, num_classes)
initial_model_state = copy.deepcopy(base_model.state_dict())
del base_model


def state_dict_fingerprint(state_dict):
    digest = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("utf-8"))
        digest.update(np.asarray(tensor.shape, dtype=np.int64).tobytes())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


initial_model_sha256 = state_dict_fingerprint(initial_model_state)


def create_model():
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(initial_model_state)
    return model


reference_history_path = Path(
    "results_resnet50_aug_alpha02_p05/histories/"
    "basic_cutmix_alpha0.2_p0.5.pt"
)
if reference_history_path.exists():
    reference_history = torch.load(
        reference_history_path,
        map_location="cpu",
    )
    assert reference_history["initial_model_sha256"] == initial_model_sha256
    assert reference_history["train_indices_sha256"] == train_indices_sha256
    assert reference_history["val_indices_sha256"] == val_indices_sha256
    print("기존 실험과 초기 모델 및 데이터 분할 hash가 일치합니다.")
else:
    print("참고 history가 없어 현재 hash만 기록합니다.")

print(f"Initial model SHA256: {initial_model_sha256}")

## 8. sample별 lam을 사용하는 학습 함수

Semantic mask 면적은 sample마다 다르므로 `CrossEntropyLoss(reduction="none")`으로
sample별 손실을 구한 뒤 각자의 `lam`으로 가중합니다.

- 요청 혼합률: `p=0.5`로 Semantic CutMix를 시도한 batch 비율
- 실제 적용률: mask가 모두 존재해 실제 합성이 수행된 batch 비율
- mask 실패 batch: 요청했지만 donor mask가 없어 일반 CE로 처리한 batch 수


In [ ]:
def evaluate(model, data_loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _ in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            losses = criterion(outputs, labels)
            batch_size = labels.size(0)
            running_loss += losses.sum().item()
            correct += outputs.argmax(dim=1).eq(labels).sum().item()
            total += batch_size

    return running_loss / total, 100.0 * correct / total


def save_best_checkpoint(model, epoch, val_loss, val_accuracy):
    checkpoint = {
        "model_state_dict": {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        },
        "architecture": "resnet50",
        "num_classes": num_classes,
        "class_names": class_names,
        "epoch": epoch,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "seed": SEED,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "augmentation": "basic",
        "batch_mix": "semantic_mask_cutmix",
        "mix_probability": MIX_PROBABILITY,
        "mix_alpha": None,
        "initial_model_sha256": initial_model_sha256,
        "train_indices_sha256": train_indices_sha256,
        "val_indices_sha256": val_indices_sha256,
        "mask_cache_config": mask_cache_config,
    }
    torch.save(checkpoint, CHECKPOINT_PATH)


def train_semantic_cutmix(model):
    set_seed(SEED)
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(reduction="none")
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)
    best_val_accuracy = float("-inf")
    history_rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        weighted_correct = 0.0
        total_samples = 0
        total_batches = 0
        requested_batches = 0
        applied_batches = 0
        mask_failure_batches = 0
        pasted_ratio_sum = 0.0
        pasted_ratio_samples = 0

        for images, labels, masks in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            total_batches += 1

            requested = random.random() < MIX_PROBABILITY
            if requested:
                requested_batches += 1
                (
                    model_images,
                    labels_a,
                    labels_b,
                    receiver_lams,
                    applied,
                ) = semantic_cutmix_batch(images, labels, masks)
                if applied:
                    applied_batches += 1
                    donor_ratios = 1.0 - receiver_lams
                    pasted_ratio_sum += donor_ratios.sum().item()
                    pasted_ratio_samples += labels.size(0)
                else:
                    mask_failure_batches += 1
            else:
                model_images = images
                labels_a = labels_b = labels
                receiver_lams = torch.ones(
                    labels.size(0),
                    device=DEVICE,
                )

            optimizer.zero_grad()
            outputs = model(model_images)
            losses_a = criterion(outputs, labels_a)
            losses_b = criterion(outputs, labels_b)
            losses = (
                receiver_lams * losses_a
                + (1.0 - receiver_lams) * losses_b
            )
            loss = losses.mean()
            loss.backward()
            optimizer.step()

            predicted = outputs.argmax(dim=1)
            batch_size = labels.size(0)
            running_loss += losses.sum().item()
            weighted_correct += (
                receiver_lams
                * predicted.eq(labels_a).float()
                + (1.0 - receiver_lams)
                * predicted.eq(labels_b).float()
            ).sum().item()
            total_samples += batch_size

        train_loss = running_loss / total_samples
        train_accuracy = 100.0 * weighted_correct / total_samples
        val_loss, val_accuracy = evaluate(model, val_loader, criterion)
        requested_mix_ratio = requested_batches / total_batches
        actual_mix_ratio = applied_batches / total_batches
        mean_pasted_mask_ratio = (
            pasted_ratio_sum / pasted_ratio_samples
            if pasted_ratio_samples
            else 0.0
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "total_batches": total_batches,
            "requested_mix_batches": requested_batches,
            "applied_mix_batches": applied_batches,
            "mask_failure_batches": mask_failure_batches,
            "requested_mix_ratio": requested_mix_ratio,
            "actual_mix_ratio": actual_mix_ratio,
            "mean_pasted_mask_ratio": mean_pasted_mask_ratio,
        }
        history_rows.append(row)
        pd.DataFrame(history_rows).to_csv(HISTORY_PATH, index=False)

        improved = val_accuracy > best_val_accuracy
        if improved:
            best_val_accuracy = val_accuracy
            save_best_checkpoint(
                model,
                epoch,
                val_loss,
                val_accuracy,
            )

        marker = " | BEST SAVED" if improved else ""
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_accuracy:.2f}% | "
            f"requested {requested_mix_ratio:.3f} | "
            f"applied {actual_mix_ratio:.3f} | "
            f"mask fail {mask_failure_batches} | "
            f"mask area {mean_pasted_mask_ratio:.3f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_accuracy:.2f}%{marker}"
        )

        if epoch % COOLDOWN_EVERY_EPOCHS == 0 and epoch < EPOCHS:
            print(f"GPU 냉각을 위해 {COOLDOWN_SECONDS}초간 대기합니다.")
            time.sleep(COOLDOWN_SECONDS)

    return pd.DataFrame(history_rows)


experiment_config = {
    "experiment": "basic_semantic_mask_cutmix_p0.5",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "SGD",
    "learning_rate": LEARNING_RATE,
    "augmentation": "basic",
    "batch_mix": "semantic_mask_cutmix",
    "mix_probability": MIX_PROBABILITY,
    "mix_alpha": None,
    "label_weight": "actual_pasted_mask_area",
    "mask_failure_policy": "skip_entire_batch_mix",
    "initial_model_sha256": initial_model_sha256,
    "train_indices_sha256": train_indices_sha256,
    "val_indices_sha256": val_indices_sha256,
}
EXPERIMENT_CONFIG_PATH.write_text(
    json.dumps(experiment_config, indent=2),
    encoding="utf-8",
)

## 9. 학습 실행 또는 checkpoint 재사용

checkpoint가 없거나 `RETRAIN=True`일 때만 20 epoch를 학습합니다.


In [ ]:
if RETRAIN or not CHECKPOINT_PATH.exists():
    semantic_model = create_model()
    assert (
        state_dict_fingerprint(semantic_model.state_dict())
        == initial_model_sha256
    )
    history_df = train_semantic_cutmix(semantic_model)

    semantic_model.to("cpu")
    del semantic_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    if not HISTORY_PATH.exists():
        raise FileNotFoundError(
            "checkpoint는 있지만 history가 없습니다. "
            "RETRAIN=True로 다시 학습하세요."
        )
    history_df = pd.read_csv(HISTORY_PATH)
    print(f"저장된 checkpoint를 재사용합니다: {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
assert checkpoint["architecture"] == "resnet50"
assert checkpoint["batch_mix"] == "semantic_mask_cutmix"
assert checkpoint["mix_alpha"] is None
assert checkpoint["mix_probability"] == MIX_PROBABILITY
assert checkpoint["class_names"] == class_names
assert checkpoint["initial_model_sha256"] == initial_model_sha256
assert checkpoint["train_indices_sha256"] == train_indices_sha256
assert checkpoint["val_indices_sha256"] == val_indices_sha256

print(
    f"Best checkpoint: epoch {checkpoint['epoch']} | "
    f"val loss {checkpoint['val_loss']:.4f} | "
    f"val acc {checkpoint['val_accuracy']:.4f}%"
)

## 10. 학습곡선과 mixing 동작 확인

validation은 원본 이미지와 hard label로 계산합니다. Semantic CutMix train accuracy는
sample별 mask 면적 가중 정확도이므로 일반 accuracy와 의미가 다릅니다.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    label="weighted train accuracy",
)
axes[1].plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    label="validation accuracy",
)
axes[1].axvline(
    checkpoint["epoch"],
    color="red",
    linestyle="--",
    label="best checkpoint",
)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(
    history_df["epoch"],
    history_df["requested_mix_ratio"],
    label="requested ratio",
)
axes[2].plot(
    history_df["epoch"],
    history_df["actual_mix_ratio"],
    label="actual applied ratio",
)
axes[2].plot(
    history_df["epoch"],
    history_df["mean_pasted_mask_ratio"],
    label="mean pasted mask area",
)
axes[2].axhline(
    MIX_PROBABILITY,
    color="black",
    linestyle=":",
    label="target p=0.5",
)
axes[2].set_title("Semantic Mixing Statistics")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.3)
axes[2].legend()

fig.suptitle("Basic + Semantic Mask CutMix")
fig.tight_layout()
plt.show()

display(
    history_df[[
        "epoch",
        "requested_mix_ratio",
        "actual_mix_ratio",
        "mask_failure_batches",
        "mean_pasted_mask_ratio",
    ]].round(4)
)

## 11. 기존 Basic CutMix 참고값과 비교

기존 실험은 Beta(`alpha=0.2`)가 사각형 면적을 결정했지만 Semantic CutMix는 DeepLab이
찾은 실제 dog mask 면적을 사용합니다. 따라서 이 비교는 동일한 영역 분포의 엄밀한
ablation이 아니라 두 mixing 전략의 결과 비교입니다.


In [ ]:
if not REFERENCE_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"기존 비교 CSV를 찾을 수 없습니다: {REFERENCE_SUMMARY_PATH}"
    )

reference_summary_df = pd.read_csv(REFERENCE_SUMMARY_PATH)
reference_row = reference_summary_df.loc[
    reference_summary_df["experiment"] == REFERENCE_EXPERIMENT
]
if len(reference_row) != 1:
    raise RuntimeError(
        f"{REFERENCE_EXPERIMENT} 행을 정확히 하나 찾지 못했습니다."
    )

reference_best_accuracy = float(
    reference_row.iloc[0]["best_val_accuracy"]
)
semantic_best_accuracy = float(checkpoint["val_accuracy"])

comparison_df = pd.DataFrame([
    {
        "condition": "Basic + random rectangle CutMix",
        "area_rule": "Beta(alpha=0.2)",
        "mix_probability": 0.5,
        "best_val_accuracy": reference_best_accuracy,
    },
    {
        "condition": "Basic + Semantic Mask CutMix",
        "area_rule": "DeepLab dog mask area",
        "mix_probability": MIX_PROBABILITY,
        "best_val_accuracy": semantic_best_accuracy,
    },
])
display(comparison_df.round(4))
comparison_df.to_csv(
    RESULT_DIR / "comparison.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(
    comparison_df["condition"],
    comparison_df["best_val_accuracy"],
    color=["#74c476", "#756bb1"],
    edgecolor="black",
)
ax.bar_label(bars, fmt="%.3f", padding=3)
minimum = comparison_df["best_val_accuracy"].min()
maximum = comparison_df["best_val_accuracy"].max()
ax.set_ylim(minimum - 0.8, maximum + 0.8)
ax.set_ylabel("Best Validation Accuracy (%)")
ax.set_title("Random Rectangle CutMix vs Semantic Mask CutMix")
ax.tick_params(axis="x", rotation=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(COMPARISON_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 12. 해석 기준과 한계

1. **객체 중심 혼합**
   - 기존 CutMix는 무작위 사각형을 교체하지만 Semantic CutMix는 dog 윤곽만 교체합니다.
   - 모델이 배경보다 강아지 특징에 집중하도록 유도할 가능성이 있습니다.

2. **alpha를 제거한 이유**
   - 영역 크기를 Beta 분포가 아니라 segmentation mask가 결정합니다.
   - 핵심 지표는 `mean_pasted_mask_ratio`와 전체 mask 면적 분포입니다.

3. **mask 실패**
   - DeepLab이 dog를 찾지 못한 donor가 하나라도 있으면 해당 batch는 일반 CE로 학습합니다.
   - 따라서 요청 혼합률과 실제 적용률이 다를 수 있습니다.

4. **semantic segmentation의 한계**
   - DeepLab은 견종을 구분하지 않고 모든 강아지를 같은 `dog` class로 처리합니다.
   - 경계가 부정확하거나 여러 강아지가 하나의 mask로 합쳐질 수 있습니다.

5. **비교의 한계**
   - 기존 CutMix와 면적 분포가 다르고 seed 42 단일 실행이므로 작은 차이를 확정적인
     우위로 해석하지 않습니다.
